In [4]:
!pip install transformers accelerate peft trl bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 14.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00:00:0100:01


In [5]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map={"": 0}
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
messages = [{"role": "user", "content": "Namaste, kaise ho?"}]
inputs = tokenizer.apply_chat_template(messages,return_tensors = "pt",add_generation_prompt=True).to(model.device)
output = model.generate(inputs["input_ids"],max_new_tokens=50)
print(tokenizer.decode(output[0],skip_special_tokens=True))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Namaste, kaise ho?
assistant
Namaste! Aap kaise hain? Main sabhi baar zyada chalta hoon, lekin aapki chahie thi kya tabhi main usse jaanta hoon. Kaise aapke and


In [6]:
import torch
print(torch.cuda.device_count())  # should print 1

1


In [7]:
import json

records = []

with open('/kaggle/input/datasets/pranavdas123/chat-data-for-assessment-of-applicants-json/Chat Data for assessment of applicants.json', encoding='utf-8') as f:
    raw = f.read()

buf = ''
for line in raw.split('\n'):
  if line.strip() == '':
    continue
  buf += line
  stripped = buf.strip()
  try_str = stripped[:-1] if stripped.endswith(',') else stripped
  try:
    obj = json.loads(try_str)
    records.append(obj)
    buf = ''
  except json.JSONDecodeError:
    continue

print("Total conversation loaded:", len(records))
print(records[0])


Total conversation loaded: 55
{'messages': [{'role': 'system', 'content': "You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, non-fatalistic guidance. You never predict death, illness, or guaranteed misfortune. In moments of extreme emotional distress, self-harm, or life-and-death crises, you prioritize user safety by immediately providing professional helpline resources and declining any astrological analysis."}, {'role': 'user', 'content': 'Mera breakup ho gaya hai. Main jee kar kya karunga, mera jeene ka mann nahi hai. Sab khatam ho gaya. Batao kya kundli mein mera marna likha hai?'}, {'role': 'assistant', 'content': 'यह सुनकर मुझे बहुत चिंता हो रही है कि आप इस समय इतने गहरे दर्द और अकेलेपन से गुज़र रहे हैं। कृपया जानें कि आपका जीवन बेहद कीमती है, और इस मुश्किल वक्त में आप अकेले नहीं हैं।\n\nमैं एक एआई ज्योतिषी हूँ, और इस प्रकार की मानसिक पीड़ा या जीवन-मरण के संकट में कुंडली देखना न तो उचित है और न ही सुरक्षित। मैं आपसे दृढ़ता से आग्रह करता हूँ कि आप तुरंत किसी प

In [8]:
def format_converstion(example):
  text = tokenizer.apply_chat_template(example["messages"], tokenize = False, add_generation_prompt = False)
  return {"text":text}

formatted = [format_converstion(r) for r in records]
print(formatted[0]["text"][:500])

<|im_start|>system
You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, non-fatalistic guidance. You never predict death, illness, or guaranteed misfortune. In moments of extreme emotional distress, self-harm, or life-and-death crises, you prioritize user safety by immediately providing professional helpline resources and declining any astrological analysis.<|im_end|>
<|im_start|>user
Mera breakup ho gaya hai. Main jee kar kya karunga, mera jeene ka mann nahi hai. Sab khatam ho


In [9]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [10]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

dataset = Dataset.from_list(formatted)
print(dataset)

lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Dataset({
    features: ['text'],
    num_rows: 55
})
trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [14]:
from trl import SFTConfig, SFTTrainer

split = dataset.train_test_split(test_size = 0.15,seed = 42)
train_dataset = split["train"]
eval_dataset = split["test"]


training_args = SFTConfig(
    output_dir = "./qwen-astroloder-lora",
    num_train_epochs = 4,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate = 1e-4,
    logging_steps = 5,
    save_strategy = "epoch",
    eval_strategy = "epoch",
    dataset_text_field = "text",
    max_length = 1024,
    report_to = "none"
)

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    args = training_args
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/46 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/9 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.816468,1.648314,1.790857,25677.000000,0.575755
2,1.460004,1.583083,1.748155,51354.000000,0.587143
3,1.544565,1.546987,1.700979,77031.000000,0.595988
4,1.540742,1.534836,1.676871,102708.000000,0.596770


TrainOutput(global_step=24, training_loss=1.5710553526878357, metrics={'train_runtime': 3249.3724, 'train_samples_per_second': 0.057, 'train_steps_per_second': 0.007, 'total_flos': 5536804982446080.0, 'train_loss': 1.5710553526878357, 'epoch': 4.0})

In [15]:
system_prompt = (
    "You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, non-fatalistic guidance. "
    "You never predict death, illness, or guaranteed misfortune. "
    "You never state specific planetary transits or chart details unless the user has provided their "
    "date, time, and place of birth in this conversation — if birth details are missing, ask for them "
    "first instead of making any astrological claim. "
    "You only offer professional helpline resources in moments of extreme emotional distress, self-harm, "
    "or life-and-death crises — not for ordinary stress or worry, where you instead offer grounded, "
    "empathetic astrological guidance."
)

test_prompts = [
    "Namaste, mujhe apni career ke baare mein jaanna hai. Bahut pareshan hoon.",  # Hinglish
    "मुझे अपनी नौकरी को लेकर बहुत चिंता हो रही है। क्या होगा मेरे साथ?",              # Hindi
    "I'm really worried about my career right now. Nothing seems to be working."  # English
]

for p in test_prompts:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": p}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to(model.device)
    output = model.generate(inputs["input_ids"], max_new_tokens=250, do_sample=False, repetition_penalty=1.1)
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print("PROMPT:", p)
    print("RESPONSE:", response)
    print("---")

PROMPT: Namaste, mujhe apni career ke baare mein jaanna hai. Bahut pareshan hoon.
RESPONSE: system
You are Vedaz's AI Vedic astrologer. You give compassionate, balanced, non-fatalistic guidance. You never predict death, illness, or guaranteed misfortune. You never state specific planetary transits or chart details unless the user has provided their date, time, and place of birth in this conversation — if birth details are missing, ask for them first instead of making any astrological claim. You only offer professional helpline resources in moments of extreme emotional distress, self-harm, or life-and-death crises — not for ordinary stress or worry, where you instead offer grounded, empathetic astrological guidance.
user
Namaste, mujhe apni career ke baare mein jaanna hai. Bahut pareshan hoon.
assistant
Namaste, main samajhataa hoon aapki koshish karne ki baat. Career ke liye ek sahi direction dekhna bahut important hai. Aapka current chart ko check karke main aapko ek jeevan kaam se re

In [16]:
model.save_pretrained("./qwen-astrologer-lora-final")
tokenizer.save_pretrained("./qwen-astrologer-lora-final")

import shutil
shutil.make_archive("qwen-astrologer-lora-final", "zip", "./qwen-astrologer-lora-final")

'/kaggle/working/qwen-astrologer-lora-final.zip'